In [ ]:
from selenium.webdriver.common.by import By
import time
from datetime import datetime,timezone
from os import name
from selenium import webdriver
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import json
import requests
import mysql.connector

import sys
import os
import pandas as pd


In [ ]:
def crear_driver_chrome(headless=True):
    # for windows
    if name == 'nt':
        PATH = r"chromedriver.exe"

    # for mac and linux(here, os.name is 'posix')
    else:
        PATH = r"./chromedriver"

    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument('--no-sandbox')
        options.add_argument('--headless')
        options.add_argument('--disable-gpu')
        options.add_argument('--disable-dev-shm-usage')
    
    options.add_experimental_option('excludeSwitches', ['enable-logging'])

    options.add_argument('--lang=es')
    options.add_experimental_option('prefs', {'intl.accept_languages': 'es,es_ES'})
    
    driver = webdriver.Chrome(executable_path=PATH, options=options)

    driver.set_window_size(1920, 1080)
    driver.maximize_window()

    return driver

In [ ]:
#Entramos en la web
driver = crear_driver_chrome(False)
driver.get("https://www.conexoo.com/")
time.sleep(3)
print('entra en web')
privacidad = driver.find_element(By.LINK_TEXT, value='Acceso Clientes')
privacidad.click()
time.sleep(1)
print('paso privacidad')
time.sleep(2)
print('entra en acceso a clientes')


time.sleep(3)

In [ ]:
driver.find_element(By.ID, "login").send_keys('***REDACTED***')
time.sleep(1.5)
driver.find_element(By.ID, 'password').send_keys('***REDACTED***')
time.sleep(3)



In [ ]:

par = driver.current_url
rellenar_resultado_recaptcha = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'g-recaptcha-response'))
)


In [ ]:
clicar = driver.find_element(By.CLASS_NAME, 'g-recaptcha')
clicar.click()
time.sleep(5)

In [ ]:
logear =driver.find_element(By.CLASS_NAME, 'btn-lg')
logear.click()
time.sleep(4)

In [ ]:
comprar_articulos = driver.find_element(By.XPATH, '//span[normalize-space()="Comprar artículos"]')
comprar_articulos.click()
time.sleep(2)


In [ ]:
tipoMedios = ['Webs y Blogs', 'Periódicos']
dataFinalArr = {}

In [ ]:
#En principio no hace falta este bloque
#webs = driver.find_element(By.XPATH, '//*[@id="sidebarnav"]/li[4]/ul/li[1]/a')#//*[@id="menubar"]/div/div/div[1]/ul/li[3]
#webs.click()
#print('pulsado en el menú las webs')
#time.sleep(2)

In [ ]:
for tipoMedio in tipoMedios:
    #Clics en webs y Blogs/Periodicos
    driver.execute_script("$('ul#sidebarnav a:contains(\"" + tipoMedio + "\")')[0].click();")
    time.sleep(5)
    
    #Clic en btn_view_list
    btn = driver.find_element(By.XPATH, '//*[@id="btn_view_list"]')
    btn.click()
    time.sleep(3)
    
    #Clic resetear filtros
    reset = driver.find_element(By.ID, 'btn_reset_filters')
    preloaderEstado = 'block'
    while(preloaderEstado!='none'):
        time.sleep(8)
        preloaderEstado = driver.execute_script("return $('#table_websites_processing').css('display')")
    print(time.strftime("%d/%m/%y - %H:%M:%S") + ' | Escaneando todos los medios de -> ' + tipoMedio)
    dataFinalArr[tipoMedio] =driver.execute_script("""let medios = []; 
                                                    $('#table_websites').DataTable().rows().data().each(function(medio){
                                                        medios.push(medio);
                                                        });
                                                    return medios;""")
    

In [ ]:
#Transformando los arrays en listas
dataFinal = []
for unDataFinal in dataFinalArr:
    dataFinal.extend(dataFinalArr[unDataFinal])

print(dataFinal)


In [ ]:
for campo_array in ['categorias_agregadas', 'tematicas_no_aceptadas']:
        for idData, data in enumerate(dataFinal):
            if len(data[campo_array]) > 0:
                dataFinal[idData][campo_array + '_arr'] = []
                for categoria in data[campo_array]:
                    dataFinal[idData][campo_array + '_arr'].append(categoria['description'])
            else:
                dataFinal[idData]['categorias_agregadas_arr'] = []

In [ ]:
dataFinal

In [ ]:
lista = []
contador = 1
for medio in dataFinal:
    for dato in ['url_link', 'descripcion', 'idioma', 'idioma_traducido', 'pais_audiencia,da', 'cf', 'dr', 'pa', 'ur', 'tf', 'rd', 'obl', 'precio_post', 'precio_post_cliente', 'max_links_post', 'tipo_sitio', 'tipo_sitio_text', 'crank', 'keywords', 'traffic_organic', 'majestic_links', 'majestic_rd,analytics', 'art_ya_pub', 'aviso_publicidad', 'bl', 'clase_web', 'clase_web_text', 'codigo_analytics', 'codigo_integracion', 'created_at', 'deleted_at', 'discard,favorite', 'fecha_act_ahrefs', 'fecha_act_majestic', 'fecha_act_sistrix,group_check', 'home_appearance', 'image', 'newsletter_difusion', 'ps_rank', 'social_publication', 'tarjetas_rojas', 'texto_conexoo', 'updated_at', 'userId,valoraciones_count', 'visitas_dia']:
        if not dato in medio:
            medio[dato] = None
    for dato in ['categorias_agregadas_arr', 'tematicas_no_aceptadas_arr']:
        if not dato in medio:
            medio[dato] = []

    lista.append(
        (
            medio['id'], 
            medio['url'], 
            medio['url_link'], 
            medio['descripcion'], 
            medio['idioma'], 
            medio['idioma_traducido'], 
            medio['pais_audiencia'], 
            medio['da'], 
            medio['cf'], 
            medio['dr'], 
            medio['pa'], 
            medio['ur'], 
            medio['tf'], 
            medio['rd'], 
            medio['obl'], 
            float(medio['precio_post']), 
            float(medio['precio_post_cliente']), 
            medio['max_links_post'], 
            medio['tipo_sitio'], 
            medio['tipo_sitio_text'], 
            medio['crank'], 
            medio['keywords'], 
            medio['traffic_organic'], 
            medio['majestic_links'], 
            medio['majestic_rd'], 
            medio['art_ya_pub'], 
            medio['aviso_publicidad'], 
            medio['bl'], 
            medio['clase_web'], 
            medio['clase_web_text'], 
            medio['created_at'], 
            medio['discard'], 
            medio['favorite'], 
            medio['group_check'], 
            medio['home_appearance'], 
            medio['image'], 
            medio['newsletter_difusion'],  
            medio['ps_rank'], 
            medio['social_publication'], 
            medio['tarjetas_rojas'], 
            medio['texto_conexoo'], 
            medio['updated_at'], 
            medio['userId'], 
            medio['valoraciones_count'], 
            medio['visitas_dia'], 
            '-|-'.join(medio['categorias_agregadas_arr']), 
            '-|-'.join(medio['tematicas_no_aceptadas_arr'])
        )
    )
    contador += 1

In [ ]:
lista

In [ ]:
columns_name = ['id','url','url_link', 'descripcion', 'idioma','idioma_traducido', 
            'pais_audiencia','da','cf','dr','pa','ur','tf','rd','obl','precio_post', 
            'precio_post_cliente','max_links_post','tipo_sitio','tipo_sitio_text', 
            'crank','keywords','traffic_organic','majestic_links','majestic_rd', 
            'art_ya_pub','aviso_publicidad','bl','clase_web','clase_web_text', 
            'created_at','discard','favorite','group_check','home_appearance','image', 
            'newsletter_difusion','ps_rank','social_publication','tarjetas_rojas','texto_conexoo', 
            'updated_at','userId','valoraciones_count','visitas_dia','categorias_agregadas_arr','tematicas_no_aceptadas_arr'] 

df = pd.DataFrame(lista, columns=columns_name)
df

In [ ]:
df.to_csv('conexoo.csv')


In [ ]:
url_link = df['url']
url_link.to_csv('urls.csv')